# PlantVillage Balance Dataset

This notebook creates a capped version of `dataset/raw/PlantVillage` using
the following rule:

- if a class has `<= 1000` images, keep all of them
- if a class has `> 1000` images, randomly keep `1000`

The output is written to:

- `dataset/raw/balance dataset`


In [ ]:
from __future__ import annotations

from pathlib import Path
import random
import shutil

import pandas as pd


In [ ]:
BASE_DIR = Path.cwd().resolve().parent
SOURCE_DIR = BASE_DIR / 'dataset' / 'raw' / 'PlantVillage'
TARGET_DIR = BASE_DIR / 'dataset' / 'raw' / 'balance dataset'
MAX_IMAGES_PER_CLASS = 1000
RANDOM_SEED = 42

assert SOURCE_DIR.exists(), SOURCE_DIR
SOURCE_DIR, TARGET_DIR


In [ ]:
def list_class_files(class_dir: Path) -> list[Path]:
    """Return sorted image files inside a class directory."""
    return sorted(path for path in class_dir.iterdir() if path.is_file())


class_file_map = {
    class_dir.name: list_class_files(class_dir)
    for class_dir in sorted(SOURCE_DIR.iterdir())
    if class_dir.is_dir()
}

source_counts_df = (
    pd.Series(
        {class_name: len(files) for class_name, files in class_file_map.items()},
        name='source_count',
    )
    .rename_axis('class_name')
    .reset_index()
)
source_counts_df


In [ ]:
rng = random.Random(RANDOM_SEED)

selected_files_by_class: dict[str, list[Path]] = {}
for class_name, files in class_file_map.items():
    if len(files) <= MAX_IMAGES_PER_CLASS:
        selected_files_by_class[class_name] = files
    else:
        selected_files_by_class[class_name] = sorted(
            rng.sample(files, MAX_IMAGES_PER_CLASS),
            key=lambda path: path.name,
        )

selection_summary_df = pd.DataFrame(
    {
        'class_name': list(selected_files_by_class.keys()),
        'selected_count': [
            len(files) for files in selected_files_by_class.values()
        ],
    }
)

selection_summary_df = source_counts_df.merge(selection_summary_df, on='class_name')
selection_summary_df['removed_count'] = (
    selection_summary_df['source_count'] - selection_summary_df['selected_count']
)
selection_summary_df.sort_values('class_name').reset_index(drop=True)


In [ ]:
if TARGET_DIR.exists():
    raise FileExistsError(
        f'{TARGET_DIR} already exists. Remove it first if you want a fresh export.'
    )

for class_name, files in selected_files_by_class.items():
    class_target_dir = TARGET_DIR / class_name
    class_target_dir.mkdir(parents=True, exist_ok=True)
    for source_file in files:
        shutil.copy2(source_file, class_target_dir / source_file.name)

print(f'Balanced dataset created at: {TARGET_DIR}')


In [ ]:
balanced_counts_df = pd.DataFrame(
    {
        'class_name': sorted(selected_files_by_class.keys()),
        'balanced_count': [
            len(selected_files_by_class[class_name])
            for class_name in sorted(selected_files_by_class.keys())
        ],
    }
)

balanced_counts_df


In [ ]:
print(f'Total classes: {len(selected_files_by_class)}')
print(f'Total images in balanced dataset: {balanced_counts_df['balanced_count'].sum():,}')
